# Build CA Utilities Dictionary

Builds `ca_utilities_dictionary.csv` — California water and electric utilities that show up in Groundwater Sustainability Plans (and in plan-adjacent surface-water, hydroelectric, and conjunctive-use contexts). Combined into one dictionary because many CA entities are *both* water and electric (LADWP, IID, MID, TID, SMUD, etc.) — splitting would force dual-purpose entities to appear in two places.

Each row has:
- `all_names`: pipe-separated full name + aliases (R pipeline splits on `|`)
- `entity_type`: utility category — see values below
- `notes`: hydro / service-area / regulatory notes

## entity_type values
- `electric_iou`  : investor-owned electric utility regulated by CPUC (PG&E, SCE, SDG&E, PacifiCorp, BVES, Liberty)
- `electric_pou`  : publicly-owned electric utility — municipal department or PUD
- `electric_jpa`  : joint powers authority for power (NCPA, SCPPA)
- `electric_coop` : member-owned rural electric cooperative
- `water_system`  : drinking water system (community water system) from the CA Waterboards SAFER list
- `water_jpa`     : water joint powers authority / inter-agency consortium
- `dual_utility`  : provides both water and electric service

## Live data sources
Two live fetches drive the bulk of the dictionary:

1. **CEC Electric Load Serving Entities (IOU & POU)** — California Energy Commission
    - About: https://gis.data.ca.gov/datasets/CAEnergy::electric-load-serving-entities-iou-pou/about
    - Item ID: `30410214d637434ba1003cbdcc32cf55`
    - Coverage: all CPUC-jurisdictional IOUs (PG&E, SCE, SDG&E, PacifiCorp, BVES, Liberty) and POUs (municipal departments, PUDs).
    - **Not covered**: JPAs (NCPA, SCPPA), member-owned cooperatives — these are in CEC's separate "Electric Load Serving Entities (Other)" dataset (https://gis.data.ca.gov/datasets/CAEnergy::electric-load-serving-entities-other). Currently handled by the hand-curated supplement below; can be wired in as a second live fetch later.

2. **CA Waterboards Drinking Water System Area Boundaries** — Division of Drinking Water (SWRCB)
    - About: https://gis.data.ca.gov/datasets/waterboards::california-drinking-water-system-area-boundaries/about
    - Item ID: `fbba842bf134497c9d611ad506ec48cc`
    - Coverage: every Public Water System with a known boundary (~3,000-7,000 systems). Filtered to **community water systems with ≥ 500 connections** so the dict isn't drowned in tiny rural mutuals.

## Hand-curated supplements (inline below)
- **Known abbreviations / aliases** for the big-name utilities — GIS feeds rarely carry these. Pipe-joined into `all_names` after the fetched canonical name.
- **Electric JPAs and cooperatives** — not in the IOU+POU feed, added as a small inline list.
- **Water JPAs** — not in the drinking water feed (they're not retail providers), added inline.

## Cross-dict overlap
This dict is intentionally **not** deduped against sibling dictionaries at build time. Each dict aims to be self-contained — a comprehensive list of its category. Cross-dict alias collisions are handled at runtime in step4's `global_dict` construction and duplicate-`from` resolution loop.

## Re-running
From `core_code/dicts/`:
```
python3 -c "import json; nb=json.load(open('build_ca_utilities_dictionary.ipynb')); exec('\n'.join(''.join(c['source']) if isinstance(c['source'],list) else c['source'] for c in nb['cells'] if c['cell_type']=='code'))"
```
Or via `jupyter nbconvert --to notebook --execute build_ca_utilities_dictionary.ipynb --inplace`.


In [ ]:
import os, re, csv, io, urllib.request
from collections import Counter, defaultdict

DICTS_DIR = "."
OUT_CSV   = os.path.join(DICTS_DIR, "ca_utilities_dictionary.csv")

# CEC Electric Load Serving Entities (IOU & POU)
ELEC_ITEM_ID = "30410214d637434ba1003cbdcc32cf55"
ELEC_URL = f"https://gis.data.ca.gov/api/download/v1/items/{ELEC_ITEM_ID}/csv?layers=0"

# CA Waterboards Drinking Water System Area Boundaries
WATER_ITEM_ID = "fbba842bf134497c9d611ad506ec48cc"
WATER_URL = f"https://gis.data.ca.gov/api/download/v1/items/{WATER_ITEM_ID}/csv?layers=0"



## Known utility abbreviations (hand-curated)

GIS feeds give us canonical full names but rarely carry the common abbreviations that GSPs actually use ("PG&E", "DWR", "MWD", etc.). This mapping is keyed on a normalized form of the fetched name and supplies the pipe-joined aliases that get prepended/appended to `all_names`. Easy to extend: add a row, re-run.

In [ ]:
# Map: normalized substring -> (display name, list of aliases, entity_type, notes)
# Lookup is case-insensitive, substring-based against the fetched utility name.
# First match wins, so list more-specific patterns first.
abbr_overrides = [
    # === Electric IOUs ===
    ("pacific gas and electric",
        "Pacific Gas and Electric Company",
        ["Pacific Gas & Electric", "PG&E", "PGE", "PG and E"],
        "electric_iou", "largest CA IOU; ~6 GW hydro on Pit, North Fork Feather, etc."),
    ("southern california edison",
        "Southern California Edison",
        ["SCE", "SoCal Edison", "Edison International", "Edison"],
        "electric_iou", "Big Creek Hydro Project; subsidiary of Edison International"),
    ("san diego gas",
        "San Diego Gas & Electric",
        ["San Diego Gas and Electric", "SDG&E", "SDGE"],
        "electric_iou", "Sempra subsidiary"),
    ("pacificorp",
        "PacifiCorp",
        ["Pacific Power", "Pacific Power & Light"],
        "electric_iou", "NE CA service territory; Klamath River dams (under removal)"),
    ("bear valley electric",
        "Bear Valley Electric Service",
        ["BVES"],
        "electric_iou", "Big Bear Lake area; American States Water subsidiary"),
    ("liberty utilities",
        "Liberty Utilities California Pacific Electric Company",
        ["Liberty Utilities", "CalPeco", "Liberty Electric"],
        "electric_iou", "Tahoe/Truckee CA service area; Algonquin Power subsidiary"),
    # === Dual-utility (water + electric) ===
    ("los angeles department of water and power",
        "Los Angeles Department of Water and Power",
        ["LADWP", "LA DWP", "LA Department of Water and Power"],
        "dual_utility", "largest US POU; LA Aqueduct + hydro"),
    ("sacramento municipal utility district",
        "Sacramento Municipal Utility District",
        ["SMUD"],
        "electric_pou", "Upper American River Project (UARP) hydro"),
    ("imperial irrigation district",
        "Imperial Irrigation District",
        ["IID", "Imperial ID"],
        "dual_utility", "largest CA ID; ~470 MW hydro on All-American Canal"),
    ("modesto irrigation district",
        "Modesto Irrigation District",
        ["MID", "Modesto ID"],
        "dual_utility", "Don Pedro and other Tuolumne projects"),
    ("turlock irrigation district",
        "Turlock Irrigation District",
        ["TID", "Turlock ID"],
        "dual_utility", "Don Pedro joint with MID"),
    ("pasadena water and power",
        "City of Pasadena Water and Power",
        ["Pasadena Water and Power", "PWP"],
        "dual_utility", "water + electric"),
    ("burbank water and power",
        "City of Burbank Water and Power",
        ["Burbank Water and Power", "BWP"],
        "dual_utility", "water + electric"),
    ("glendale water and power",
        "City of Glendale Water and Power",
        ["Glendale Water and Power", "GWP"],
        "dual_utility", "water + electric"),
    ("anaheim public utilities",
        "City of Anaheim Public Utilities",
        ["Anaheim Public Utilities", "APU"],
        "dual_utility", "water + electric"),
    ("riverside public utilities",
        "City of Riverside Public Utilities",
        ["Riverside Public Utilities", "RPU"],
        "dual_utility", "water + electric"),
    ("truckee donner public utility",
        "Truckee Donner Public Utility District",
        ["Truckee Donner PUD", "TDPUD"],
        "dual_utility", "water + electric serving Truckee"),
    ("azusa light and water",
        "Azusa Light and Water",
        ["Azusa Light & Water"],
        "dual_utility", "water + electric"),
    ("corona",  # narrow to City of Corona
        "Corona Department of Water and Power",
        ["Corona Power", "Corona Electric"],
        "dual_utility", "water + electric"),
    # === Major water IOUs ===
    ("california water service",
        "California Water Service Company",
        ["California Water Service", "Cal Water", "CalWater", "CWS"],
        "water_iou", "Class A; largest CA water IOU"),
    ("golden state water",
        "Golden State Water Company",
        ["Golden State Water", "GSWC", "American States Water Company", "AWR"],
        "water_iou", "subsidiary of American States Water"),
    ("california-american water",
        "California-American Water Company",
        ["California American Water", "Cal Am", "Cal-Am", "CalAm"],
        "water_iou", "subsidiary of American Water"),
    ("san jose water company",
        "San Jose Water Company",
        ["San Jose Water", "SJW Company", "SJW Group"],
        "water_iou", "Class A; Santa Clara County"),
    ("san gabriel valley water",
        "San Gabriel Valley Water Company",
        ["SGVWC", "SGVW"],
        "water_iou", "Class A; San Gabriel Valley"),
    ("suburban water systems",
        "Suburban Water Systems",
        ["Suburban Water", "SWS"],
        "water_iou", "Class A; San Gabriel Valley"),
    # === Major water districts also in water_entity_dictionary — but we keep them here per the no-sibling-dedup policy ===
    ("metropolitan water district",
        "Metropolitan Water District of Southern California",
        ["Metropolitan Water District", "MWD", "MET"],
        "water_jpa", "largest CA wholesaler"),
    ("east bay municipal utility",
        "East Bay Municipal Utility District",
        ["EBMUD"],
        "dual_utility", "water; some hydro on Mokelumne"),
    ("san francisco public utilities",
        "San Francisco Public Utilities Commission",
        ["SFPUC", "SF PUC"],
        "dual_utility", "water + Hetch Hetchy hydro"),
    ("orange county water district",
        "Orange County Water District",
        ["OCWD"],
        "water_system", "Orange County groundwater management"),
    ("santa clara valley water",
        "Santa Clara Valley Water District",
        ["Valley Water", "SCVWD"],
        "water_system", "SCC water + flood"),
]

def find_abbr(name):
    """Returns (canonical_display, [aliases], entity_type_override, notes) or None."""
    nlow = (name or "").lower()
    for pat, disp, aliases, etype, notes in abbr_overrides:
        if pat in nlow:
            return disp, aliases, etype, notes
    return None


## Fetch: Electric Load Serving Entities (IOU & POU)

CEC's combined dataset for investor-owned and publicly-owned electric utilities. We probe for common name/ownership column names since the schema isn't documented in a stable place.

In [ ]:
print(f"Fetching electric LSEs from {ELEC_URL}")
with urllib.request.urlopen(ELEC_URL) as r:
    elec_bytes = r.read()
print(f"  -> {len(elec_bytes):,} bytes")
elec_rows = list(csv.DictReader(io.StringIO(elec_bytes.decode("utf-8-sig"))))
print(f"  -> {len(elec_rows)} records")
if elec_rows:
    print(f"  -> columns: {list(elec_rows[0].keys())}")

def first_col_ci(row, candidates):
    """Case-insensitive lookup of the first candidate column that exists."""
    lower_map = {k.lower(): k for k in row.keys()}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    return None

# Common name/type column conventions across CA Open Data utility datasets
NAME_COLS = ["Utility", "UTILITY", "Utility_Name", "UTILITY_NAME",
              "Name", "NAME", "LSE_Name", "LSE_NAME",
              "Owner", "OWNER", "OPERATOR", "OPERATOR_NAME"]
TYPE_COLS = ["Type", "TYPE", "Ownership", "OWNERSHIP",
              "Class", "CLASS", "LSE_Type", "LSE_TYPE",
              "Util_Type", "UTIL_TYPE"]

name_col = first_col_ci(elec_rows[0], NAME_COLS) if elec_rows else None
type_col = first_col_ci(elec_rows[0], TYPE_COLS) if elec_rows else None
if not name_col:
    raise RuntimeError(f"Could not find a name column. Available: {list(elec_rows[0].keys()) if elec_rows else []}")
print(f"  -> using name_col={name_col!r}, type_col={type_col!r}")


In [ ]:
# Build the electric utility rows. One row per distinct fetched name.
electric_rows = []
seen_elec = set()
for r in elec_rows:
    name = (r.get(name_col) or "").strip()
    if not name or name.lower() in seen_elec:
        continue
    seen_elec.add(name.lower())
    raw_type = (r.get(type_col) or "").strip().lower() if type_col else ""
    hit = find_abbr(name)
    if hit:
        disp, aliases, etype, notes = hit
        all_names = "|".join([disp] + [a for a in aliases if a.lower() != disp.lower()])
    else:
        all_names = name
        # Map fetched ownership type to our entity_type values
        if "iou" in raw_type or "investor" in raw_type:
            etype = "electric_iou"
        elif "pou" in raw_type or "public" in raw_type or "municipal" in raw_type:
            etype = "electric_pou"
        else:
            etype = "electric_pou"  # CEC dataset is IOU+POU only, default to POU
        notes = f"sourced from CEC LSE dataset ({raw_type})" if raw_type else "sourced from CEC LSE dataset"
    electric_rows.append((all_names, etype, notes))

print(f"electric rows: {len(electric_rows)}")
for et, n in Counter(r[1] for r in electric_rows).most_common():
    print(f"  {et:20s} {n}")


## Fetch: Drinking Water System Area Boundaries

CA Waterboards SAFER list of every public water system with a defined service area boundary. Every row's `WATER_SYSTEM_NAME` becomes a dict entry — no size or type filtering. Dedup-at-runtime handles overlap with other dicts.

In [ ]:
print(f"Fetching drinking-water systems from {WATER_URL}")
with urllib.request.urlopen(WATER_URL) as r:
    water_bytes = r.read()
print(f"  -> {len(water_bytes):,} bytes")
water_rows_raw = list(csv.DictReader(io.StringIO(water_bytes.decode("utf-8-sig"))))
print(f"  -> {len(water_rows_raw)} records")

# Only WATER_SYSTEM_NAME matters. No filtering — the dict aims to be
# comprehensive within its category; dedup is handled at runtime.
water_rows = []
seen_water = set()
for r in water_rows_raw:
    name = (r.get("WATER_SYSTEM_NAME") or "").strip()
    if not name:
        continue
    nlow = name.lower()
    if nlow in seen_water:
        continue
    seen_water.add(nlow)
    hit = find_abbr(name)
    if hit:
        disp, aliases, etype, notes = hit
        all_names = "|".join([disp] + [a for a in aliases if a.lower() != disp.lower()])
    else:
        all_names = name
        etype = "water_system"
        notes = ""
    water_rows.append((all_names, etype, notes))

print(f"kept {len(water_rows)} unique water systems")


## Hand-curated supplement

Things the live feeds don't carry: electric JPAs (NCPA, SCPPA, etc.) aren't in the IOU+POU dataset because they're not retail load-serving entities. Rural electric cooperatives and water JPAs are also missing. Adding them inline.

In [ ]:
supplement = [
    # Electric JPAs
    ("Northern California Power Agency|NCPA",
        "electric_jpa", "member POUs share generation incl. North Fork Stanislaus hydro"),
    ("Southern California Public Power Authority|SCPPA",
        "electric_jpa", "member POUs incl. LADWP, Anaheim, Pasadena, Burbank, Glendale, Riverside, Vernon"),
    ("Transmission Agency of Northern California|TANC",
        "electric_jpa", "transmission-only JPA"),
    ("M-S-R Public Power Agency|M-S-R Power Agency",
        "electric_jpa", "Modesto-Santa Clara-Redding"),
    ("Modesto-San Joaquin Public Power Agency",
        "electric_jpa", "MID/SSJID/Oakdale ID consortium"),
    # Electric cooperatives
    ("Plumas-Sierra Rural Electric Cooperative|Plumas-Sierra REC|PSREC",
        "electric_coop", "Plumas, Sierra, Lassen counties"),
    ("Surprise Valley Electrification Corporation|Surprise Valley Electric|SVEC",
        "electric_coop", "Modoc County"),
    ("Anza Electric Cooperative|Anza Electric",
        "electric_coop", "Riverside County mountain communities"),
    ("Trinity Public Utilities District|Trinity PUD|TPUD",
        "electric_coop", "Trinity County"),
    # Water JPAs
    ("Friant Water Authority|FWA",
        "water_jpa", "Friant Division CVP contractors"),
    ("San Luis and Delta-Mendota Water Authority|San Luis & Delta-Mendota Water Authority|SLDMWA|San Luis Delta Mendota",
        "water_jpa", "south-of-Delta CVP contractors"),
    ("State Water Contractors|SWC",
        "water_jpa", "29 SWP contractors"),
    ("State Water Project Contractors Authority|SWPCA",
        "water_jpa", ""),
    ("Central Valley Project Water Association|CVPWA",
        "water_jpa", "CVP contractor association"),
    ("Kings River Conservation District|KRCD",
        "water_jpa", "Kings River basin"),
    ("Kings River Water Association|Kings River Water Assoc",
        "water_jpa", ""),
    ("Mountain Counties Water Resources Association|MCWRA",
        "water_jpa", "Sierra foothills agencies"),
    ("Association of California Water Agencies|ACWA",
        "water_jpa", "statewide trade association"),
    ("California Municipal Utilities Association|CMUA",
        "water_jpa", "trade assoc for CA POU electric + water"),
    # Irrigation districts with hydro (also in water_entity_dictionary; kept per policy)
    ("Yuba County Water Agency|Yuba Water Agency|YCWA",
        "irrigation_district_with_hydro", "Yuba River Development Project"),
    ("Placer County Water Agency|PCWA",
        "irrigation_district_with_hydro", "Middle Fork American hydro"),
    ("El Dorado Irrigation District|EID",
        "irrigation_district_with_hydro", "El Dorado Hydroelectric Project"),
    ("Nevada Irrigation District|NID",
        "irrigation_district_with_hydro", "Yuba-Bear hydro project"),
    ("Merced Irrigation District|Merced ID",
        "irrigation_district_with_hydro", "New Exchequer / Lake McClure"),
    ("South San Joaquin Irrigation District|SSJID",
        "irrigation_district_with_hydro", "Tri-Dam Project"),
    ("Oakdale Irrigation District|Oakdale ID|OID",
        "irrigation_district_with_hydro", "Tri-Dam Project"),
    ("Calaveras County Water District|Calaveras County WD",
        "irrigation_district_with_hydro", "Spicer Meadow / N Fork Stanislaus"),
]
print(f"supplement: {len(supplement)} hand-curated rows")


## Combine and write

All rows from the live feeds + supplement go to the output. No sibling-dedup.

In [ ]:
all_rows = electric_rows + water_rows + supplement

with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["all_names", "entity_type", "notes"])
    for row in all_rows:
        w.writerow(row)

print(f"wrote {OUT_CSV} — {len(all_rows)} rows")
print("  by entity_type:")
for t, n in Counter(r[1] for r in all_rows).most_common():
    print(f"    {t:35s} {n}")
